##### EPMT Data "Cleanup" - Next Idea No. "11" (More data - 100k+ rows)

Following up from EPMTDataCleanup_Next7.ipynb<br>

In [1]:
%pip install pandas matplotlib seaborn scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pprint
import ast
from datetime import datetime

%matplotlib inline 

# Suggested by Gemini -- "Makes your plots look a bit more modern/clean"
sns.set_theme(style="whitegrid")

# Load some data and revisit what we have so far

In [3]:
import glob
import os

# NOTE: OLD DATA has extra data compared to the data being processed today --
#data_path = './data_40000_20260207_083037' 

# NOTE: NEW DATA was extracted from epmt database using different script --
data_path = './data_133872_20260319_071902'

all_files = glob.glob(os.path.join(data_path, "*.csv"))

df_list = [pd.read_csv(f) for f in all_files]
df = pd.concat(df_list, ignore_index=True)
print(f"Loaded {len(all_files)} files.")
pprint.pprint(all_files)

Loaded 27 files.
['./data_133872_20260319_071902/data_133872_1_20260319_071902.csv',
 './data_133872_20260319_071902/data_133872_0_20260319_071902.csv',
 './data_133872_20260319_071902/data_133872_3_20260319_071902.csv',
 './data_133872_20260319_071902/data_133872_2_20260319_071902.csv',
 './data_133872_20260319_071902/data_133872_4_20260319_071902.csv',
 './data_133872_20260319_071902/data_133872_5_20260319_071902.csv',
 './data_133872_20260319_071902/data_133872_6_20260319_071902.csv',
 './data_133872_20260319_071902/data_133872_7_20260319_071902.csv',
 './data_133872_20260319_071902/data_133872_22_20260319_071902.csv',
 './data_133872_20260319_071902/data_133872_23_20260319_071902.csv',
 './data_133872_20260319_071902/data_133872_8_20260319_071902.csv',
 './data_133872_20260319_071902/data_133872_20_20260319_071902.csv',
 './data_133872_20260319_071902/data_133872_9_20260319_071902.csv',
 './data_133872_20260319_071902/data_133872_21_20260319_071902.csv',
 './data_133872_20260319_07

In [4]:
df.columns.tolist()

['jobid',
 'env_dict',
 'annotations',
 'exitcode',
 'cpu_time',
 'duration',
 'start',
 'end',
 'tags']

In [5]:
len(df)

133872

# Trying to pull out similar data to the previous notebooks. 

Features include:
* 'exp_time' - in annotations
* 'SLURM_JOB_ACCOUNT_*' - in env_dict
* 'exp_component_*' - in annotations
* 'exp_fre_mod' - in annotations, bronx-23 for now
* 'exp_name_*' - in annotations
* 'exp_platform_*' - in annotations
* 'exp_target_*' - in annotations

Target: 
* 'cpu_time'

# Pull Out the Data (Desired Columns)
This time, before we do any standardization or take the log, we'll first just pull out all of the column data we want, including one-hot encoding. Then, only after that, will we do standardization/normalization/etc.

In [6]:
df_clean = df.copy()

In [7]:
df_clean['env_dict_eval'] = df_clean['env_dict'].apply(ast.literal_eval)

In [8]:
df_clean['SLURM_JOB_ACCOUNT'] = df_clean['env_dict_eval'].apply(lambda data: data.get('SLURM_JOB_ACCOUNT'))

In [9]:
df_clean = df_clean.drop(columns=['env_dict', 'env_dict_eval'])

In [10]:
# Turn all of the annotations from strings to dictionaries to process easier
df_clean['annotations_eval'] = pd.json_normalize(df_clean["annotations"].apply(ast.literal_eval))
df_clean['annotations_eval']

0         exp_component:atmos_level;exp_fre_mod:/home/fm...
1         exp_component:atmos;exp_fre_mod:/home/fms/loca...
2         exp_component:refineDiag;exp_fre_mod:/home/fms...
3         exp_component:atmos_month_aer;exp_fre_mod:/hom...
4         exp_component:atmos_level_cmip;exp_fre_mod:/ho...
                                ...                        
133867    exp_component:refineDiag;exp_fre_mod:/home/fms...
133868    exp_component:refineDiag;exp_fre_mod:/home/fms...
133869    exp_component:refineDiag;exp_fre_mod:/home/fms...
133870    exp_component:refineDiag;exp_fre_mod:/home/fms...
133871    exp_component:refineDiag;exp_fre_mod:/home/fms...
Name: annotations_eval, Length: 133872, dtype: str

In [11]:
# Now, expand the EPMT_JOB_TAGS strings into dictionaries
def parse_key_val_str(kv_str: str):
    if not isinstance(kv_str, str) or not kv_str:
        return {}
    return dict(item.split(":") for item in kv_str.split(";"))

df_clean["parsed_EPMT_JOB_TAGS"] = df_clean['annotations_eval'].apply(parse_key_val_str)
df_clean["parsed_EPMT_JOB_TAGS"]

0         {'exp_component': 'atmos_level', 'exp_fre_mod'...
1         {'exp_component': 'atmos', 'exp_fre_mod': '/ho...
2         {'exp_component': 'refineDiag', 'exp_fre_mod':...
3         {'exp_component': 'atmos_month_aer', 'exp_fre_...
4         {'exp_component': 'atmos_level_cmip', 'exp_fre...
                                ...                        
133867    {'exp_component': 'refineDiag', 'exp_fre_mod':...
133868    {'exp_component': 'refineDiag', 'exp_fre_mod':...
133869    {'exp_component': 'refineDiag', 'exp_fre_mod':...
133870    {'exp_component': 'refineDiag', 'exp_fre_mod':...
133871    {'exp_component': 'refineDiag', 'exp_fre_mod':...
Name: parsed_EPMT_JOB_TAGS, Length: 133872, dtype: object

In [12]:
# Turn the dictionaries into columns with values using pd.Series
# Join the parsed EPMT tags to the expanded annotations
df_clean = df_clean.join(df_clean["parsed_EPMT_JOB_TAGS"].apply(pd.Series))

In [13]:
df_clean = df_clean.drop(columns=['annotations', 'annotations_eval', 'parsed_EPMT_JOB_TAGS'])

In [14]:
df_clean.columns.tolist()

['jobid',
 'exitcode',
 'cpu_time',
 'duration',
 'start',
 'end',
 'tags',
 'SLURM_JOB_ACCOUNT',
 'exp_component',
 'exp_fre_mod',
 'exp_name',
 'exp_time',
 'exp_platform',
 'exp_target',
 'exp_seg_months',
 'script_name']

In [15]:
len(df_clean)

133872

In [16]:
print(df_clean.isna().sum())

jobid                    0
exitcode                 0
cpu_time              7448
duration                 0
start                    0
end                      0
tags                     0
SLURM_JOB_ACCOUNT        0
exp_component          926
exp_fre_mod          17702
exp_name               926
exp_time               926
exp_platform           926
exp_target             926
exp_seg_months       17702
script_name            926
dtype: int64


In [17]:
df_clean = df_clean.dropna()
len(df_clean)

108959

In [18]:
len(df_clean)

108959

## Keep Bronx-23

Discard the rows if they're not Bronx-23.

In [19]:
df_clean = df_clean[df_clean['exp_fre_mod'].str.contains('bronx-23', case=False, na=False)]
df_clean = df_clean.drop(columns=['exp_fre_mod'])

In [20]:
len(df_clean)

106447

In [21]:
df_clean.columns.tolist()

['jobid',
 'exitcode',
 'cpu_time',
 'duration',
 'start',
 'end',
 'tags',
 'SLURM_JOB_ACCOUNT',
 'exp_component',
 'exp_name',
 'exp_time',
 'exp_platform',
 'exp_target',
 'exp_seg_months',
 'script_name']

## Remove Some More Columns

In [22]:
df_clean = df_clean.drop(columns=['script_name', 'jobid', 'duration', 'start', 'end', 'tags'])

In [23]:
df_clean.columns.tolist()

['exitcode',
 'cpu_time',
 'SLURM_JOB_ACCOUNT',
 'exp_component',
 'exp_name',
 'exp_time',
 'exp_platform',
 'exp_target',
 'exp_seg_months']

## Only Keep Rows with exitcode == 0

In [24]:
df_clean = df_clean[df_clean['exitcode'] == 0]

In [25]:
len(df_clean)

106447

In [26]:
df_clean = df_clean.drop(columns=['exitcode'])

In [27]:
df_clean.columns.tolist()

['cpu_time',
 'SLURM_JOB_ACCOUNT',
 'exp_component',
 'exp_name',
 'exp_time',
 'exp_platform',
 'exp_target',
 'exp_seg_months']

## Save What I have for now

In [29]:
date_str = datetime.today().strftime("%Y%m%d_%H%M%S")
df_clean.to_csv(f"epmt_data_cleaned_{date_str}.csv", index=False)

In [31]:
print(f"Saved epmt_data_cleaned_{date_str}.csv")

Saved epmt_data_cleaned_20260326_074338.csv
